In [1]:
#| default_exp hyperparameter_tuning

## Hyperparameter Tuning

Peshbeen provides two powerful tools for hyperparameter tuning: `hyperopt_tune` and `optuna_tune`. These functions allow you to optimize the hyperparameters of your forecasting models using the `hyperopt` and `optuna` libraries, respectively. Both functions support cross-validation and can be used with any of the forecasting models available in peshbeen.

### hyperopt_tune example for univariate forecasting using machine learning models

In [2]:
from peshbeen.datasets import load_wales_admissions
from peshbeen.metrics import RMSE
from lightgbm import LGBMRegressor
from peshbeen.models import ml_forecaster
from peshbeen.model_selection import hyperopt_tune
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown="ignore")

load_wales_admissions["day_of_week"] = load_wales_admissions.index.dayofweek
load_wales_admissions["month"] = load_wales_admissions.index.month
# split the data into train and test sets
train = load_wales_admissions[:-30]
test = load_wales_admissions[-30:]
cat_variables = ["day_of_week", "month"]
# import linear regression from sklearn
ml_model = ml_forecaster(model=LGBMRegressor(verbose=-1),
              target_col='admissions', lags = 30,
              cat_variables=cat_variables, categorical_encoder=ohe)
ml_model.fit(train)

# Define the hyperparameter search space for LightGBM
from hyperopt import hp
from hyperopt.pyll import scope
lgb_param_space={'learning_rate': hp.uniform('learning_rate', 0.001, 0.6),
            'num_leaves': scope.int(hp.quniform('num_leaves', 10, 200, 1)),
           'max_depth':scope.int(hp.quniform('max_depth', 2, 18, 1)),
            'bagging_fraction': hp.uniform('bagging_fraction', 0.5, 1),
            'feature_fraction': hp.uniform('feature_fraction', 0.5, 1),
            'lambda_l2' : hp.uniform('lambda_l2', 0,10),
           'lambda_l1' : hp.uniform('lambda_l1', 0, 10),
           'top_rate' : hp.quniform('top_rate', 0.05, 0.4, 0.0001),
            'other_rate' : hp.quniform('other_rate', 0.05, 0.3, 0.0001),
           'num_iterations': scope.int(hp.quniform("num_iterations", 30, 700, 1)),
           'lags': hp.choice("lags", [
                                 [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14],
                                 [1,2,3,4,5,6,7,14,21],
                                 [1,2,3],
                             ]),
                "seed":0,
                "box_cox": hp.uniform("box_cox", 0.0, 4),
                "box_cox_biasadj": hp.choice("box_cox_biasadj", [True, False])}

# Run hyperparameter tuning using hyperopt
best_params, best_lags, other_ = hyperopt_tune(model=ml_model, df=train, cv_split=5, step_size=10,
                                        test_size=1, eval_metric=RMSE, eval_num=10,
                                        param_space=lgb_param_space)

print("Best params:", best_params)
print("Best lags:", best_lags)
print("Other info:", other_)

100%|██████████| 10/10 [00:35<00:00,  3.52s/trial, best loss: 48.92792086953741]
Best params: {'bagging_fraction': 0.7446432971714252, 'feature_fraction': 0.945825682573253, 'lambda_l1': 1.3320978418814078, 'lambda_l2': 3.93214856724596, 'learning_rate': 0.368784525651245, 'max_depth': 14, 'num_iterations': 270, 'num_leaves': 163, 'other_rate': 0.1602, 'seed': 0, 'top_rate': 0.3996}
Best lags: [1, 2, 3, 4, 5, 6, 7]
Other info: {'box_cox': 1.973823501436872, 'box_cox_biasadj': False}


In [3]:
# now we can run our model with the best paramaters, best lags and other info such as box_cox and box_cox_biasadj
best_box_cox = other_["box_cox"]
best_box_cox_biasadj = other_["box_cox_biasadj"]

ml_model = ml_forecaster(model=LGBMRegressor(**best_params, verbose=-1),
              target_col='admissions', lags = list(best_lags), box_cox=best_box_cox, box_cox_biasadj=best_box_cox_biasadj,
              cat_variables=cat_variables, categorical_encoder=ohe)
ml_model.fit(train)
ml_forecasts = ml_model.forecast(H=30, exog=test[cat_variables])
ml_forecasts

array([8841.33467931, 8693.80377381, 8695.12199759, 8896.17457136,
       8846.40941508, 8853.20652116, 8822.74780505, 8776.89831377,
       8700.44490716, 8688.37859875, 8681.35341052, 8727.78034881,
       8761.324051  , 8796.24502015, 8804.46983694, 8747.39882218,
       8726.57467281, 8755.71624434, 8777.92704472, 8790.0136147 ,
       8795.77020796, 8776.9454295 , 8679.96571371, 8673.99598484,
       8723.90934339, 8738.72871404, 8795.15837905, 8797.14670256,
       8786.28848301, 8613.1009315 ])

### optuna_tune example for univariate forecasting

In [4]:
from peshbeen.datasets import load_wales_admissions
from peshbeen.metrics import MAE, RMSE
from lightgbm import LGBMRegressor
from peshbeen.models import ml_forecaster
from peshbeen.model_selection import optuna_tune
load_wales_admissions["day_of_week"] = load_wales_admissions.index.dayofweek
load_wales_admissions["month"] = load_wales_admissions.index.month
# split the data into train and test sets
train = load_wales_admissions[:-30]
test = load_wales_admissions[-30:]
cat_variables = ["day_of_week", "month"]
# import linear regression from sklearn
ml_model = ml_forecaster(model=LGBMRegressor(verbose=-1),
              target_col='admissions', lags = 30,
              cat_variables=cat_variables, categorical_encoder=ohe)
ml_model.fit(train)
# ml_model.data_prep(train)
forecasts = ml_model.forecast(H=30, exog=test[cat_variables])
lgb_param_space = {
    "learning_rate":     lambda t: t.suggest_float("learning_rate", 0.001, 0.6),
    "num_leaves":        lambda t: t.suggest_int("num_leaves", 10, 200),
    "max_depth":         lambda t: t.suggest_int("max_depth", 2, 18),
    "bagging_fraction":  lambda t: t.suggest_float("bagging_fraction", 0.5, 1.0),
    "feature_fraction":  lambda t: t.suggest_float("feature_fraction", 0.5, 1.0),
    "lambda_l2":         lambda t: t.suggest_float("lambda_l2", 0.0, 10.0),
    "lambda_l1":         lambda t: t.suggest_float("lambda_l1", 0.0, 10.0),
    "top_rate":          lambda t: t.suggest_float("top_rate", 0.05, 0.4),
    "other_rate":        lambda t: t.suggest_float("other_rate", 0.05, 0.3),
    "num_iterations":    lambda t: t.suggest_int("num_iterations", 30, 700),
    "lags":              lambda t: t.suggest_categorical(
                             "lags", [
                                 [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14],
                                 [1,2,3,4,5,6,7,14,21],
                                 [1,2,3],
                             ]),
                             
    "seed":              lambda t: 0,   # fixed, not sampled
}

best_params, best_lags, other_ = optuna_tune(
    model=ml_model,
    df=train,
    cv_split=10,
    step_size=10,
    test_size=30,
    eval_metric=RMSE,
    eval_num=10,
    param_space=lgb_param_space, verbose=False
)
print("Best params:", best_params)
print("Best lags:", best_lags)

Best params: {'learning_rate': 0.01012470003273782, 'num_leaves': 34, 'max_depth': 10, 'bagging_fraction': 0.571405210429277, 'feature_fraction': 0.53564234644381, 'lambda_l2': 9.899362945946478, 'lambda_l1': 9.95793367914908, 'top_rate': 0.3447405358621952, 'other_rate': 0.2567356100878523, 'num_iterations': 559}
Best lags: [1, 4, 7]


### Selecting the best ETS (Error, Trend, Seasonality) model usig `optuna_tune` or `hyperopt_tune`

In [5]:
from peshbeen.models import ets

ets_param_space = {
    "smoothing_level":     lambda t: t.suggest_float("smoothing_level", 0.001, 0.99),
    "trend":              lambda t: t.suggest_categorical(
                             "trend", [
                                 "add",
                                 "mul",
                                 None
                             ]),
    "seasonal":           lambda t: t.suggest_categorical(
                             "seasonal", [
                                 "add",
                                 "mul",
                                 None
                             ]),
    "smoothing_trend":    lambda t: t.suggest_float("smoothing_trend", 0.001, 0.99),
    "smoothing_seasonal": lambda t: t.suggest_float("smoothing_seasonal", 0.001, 0.99),
    "smoothing_level":     lambda t: t.suggest_float("smoothing_level", 0.001, 0.99),
    "seasonal_periods":              lambda t: 7,   # fixed, not sampled
        "box_cox":           lambda t: t.suggest_float("box_cox", 0.0, 4),
        "box_cox_biasadj":   lambda t: t.suggest_categorical("box_cox_biasadj", [True, False])
}

ets_model = ets(target_col='admissions')
best_params, _, other_ = optuna_tune(
    model=ets_model,
    df=train,
    cv_split=4,
    step_size=1,
    test_size=30,
    eval_metric=RMSE,
    eval_num=100,
    param_space=ets_param_space, verbose=False
)
print("Best params:", best_params)
print("Other info:", other_)

Best params: {'smoothing_level': 0.03754870921005382, 'trend': None, 'seasonal': None, 'smoothing_trend': 0.8469028803723587, 'smoothing_seasonal': 0.922032619195668}
Other info: {'box_cox': 1.8437817093744235, 'box_cox_biasadj': True}


In [6]:
# now forecast wit the best parameters
best_params.update(other_)
ets_model = ets(target_col='admissions', **best_params)
ets_model.fit(train)
ets_forecasts = ets_model.forecast(H=30)

In [7]:
# get cross validation results with the best parameters
cv_df = ets_model.cross_validate(
    df=train,
    cv_split=5,
        step_size=7,
        test_size=30,
        metrics=[RMSE, MAE])
cv_df.head()

,cutoff,index,split,y_true,y_pred
0,2022-12-07,2022-12-07,fold_1,8933,8918.964691
1,2022-12-07,2022-12-08,fold_1,9013,8918.964691
2,2022-12-07,2022-12-09,fold_1,9000,8918.964691
3,2022-12-07,2022-12-10,fold_1,8821,8918.964691
4,2022-12-07,2022-12-11,fold_1,8835,8918.964691


### Selecting the best orders for an ARIMA model using `optuna_tune` or `hyperopt_tune`

In [8]:
from peshbeen.models import arima
from itertools import product
# Define the hyperparameter search space for ARIMA
p_values = [0, 1, 2, 3]
d_values = [1]
q_values = [0, 1, 2, 3]

# create the list of orders using the product of p, d and q values
orders = list(product(p_values, d_values, q_values))

# Define the hyperparameter search space for seasonal ARIMA
P_values = [0, 1, 2, 3]
D_values = [0, 1]
Q_values = [0, 1, 2, 3]

# create the list of seasonal orders using the product of P, D and Q values
seasonal_orders = list(product(P_values, D_values, Q_values))

# let's define the hyperparameter space for arima using hyperopt
arima_param_space = {
    "order": hp.choice("order", orders),
    "seasonal_order": hp.choice("seasonal_order", seasonal_orders),
    "seasonal_length": 7,
    "box_cox": hp.uniform("box_cox", 0.0, 4),
    "box_cox_biasadj": hp.choice("box_cox_biasadj", [True, False])
}

arima_model = arima(target_col='admissions')
best_params, _, other_ = hyperopt_tune(
    model=arima_model,
    df=train,
    cv_split=10,
    step_size=10,
    test_size=30,
    eval_metric=RMSE,
    eval_num=5,
    param_space=arima_param_space
)

100%|██████████| 5/5 [00:13<00:00,  2.63s/trial, best loss: 127.00351539027417]


### hyperopt_tune example for multivariate forecasting

In [9]:
from peshbeen.datasets import load_admission_calls
from peshbeen.models import ml_mv_forecaster
from peshbeen.metrics import RMSE
from peshbeen.model_selection import mv_hyperopt_tune
from lightgbm import LGBMRegressor
load_admission_calls["day_of_week"] = load_admission_calls.index.dayofweek
load_admission_calls["month"] = load_admission_calls.index.month
train = load_admission_calls[:-30]
test = load_admission_calls[-30:]

cat_variables = ["day_of_week", "month"]
mv_model = ml_mv_forecaster(model=LGBMRegressor(verbose=-1),
              target_cols=['admissions', "calls"], lags = {"admissions": 7, "calls": 7},
                cat_variables=cat_variables,
                difference={"admissions": 1, "calls": 1}, categorical_encoder=ohe)
lgb_param_space={'learning_rate': hp.uniform('learning_rate', 0.001, 0.6),
            'num_leaves': scope.int(hp.quniform('num_leaves', 10, 200, 1)),
           'max_depth':scope.int(hp.quniform('max_depth', 2, 18, 1)),
            'bagging_fraction': hp.uniform('bagging_fraction', 0.5, 1),
            'feature_fraction': hp.uniform('feature_fraction', 0.5, 1),
           'min_data_in_leaf': scope.int(hp.quniform ('min_data_in_leaf', 5, 100, 1)), 
            'lambda_l2' : hp.uniform('lambda_l2', 0,10),
           'lambda_l1' : hp.uniform('lambda_l1', 0, 10),
            'other_rate' : hp.quniform('other_rate', 0.05, 0.3, 0.0001),
           'num_iterations': scope.int(hp.quniform("num_iterations", 30, 700, 1)),
           'top_k': scope.int(hp.quniform('top_k', 8, 30, 1)),
                "seed":0, 'lags': hp.choice("lags", [
                                 [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14]])}
best_params, best_lags = mv_hyperopt_tune(model=mv_model, df=train, target_col= "admissions", cv_split=5, step_size=10,
                                        test_size=30, eval_metric=RMSE, eval_num=4,
                                        param_space=lgb_param_space)
print("Best params:", best_params)
print("Best lags:", best_lags)

100%|██████████| 4/4 [00:07<00:00,  1.95s/trial, best loss: 147.69858779669912]
Best params: {'bagging_fraction': 0.9073112490061555, 'feature_fraction': 0.5296277456248977, 'lambda_l1': 0.1550608968353795, 'lambda_l2': 7.215914257246618, 'learning_rate': 0.009035941209911674, 'max_depth': 13, 'min_data_in_leaf': 78, 'num_iterations': 374, 'num_leaves': 192, 'other_rate': 0.2501, 'seed': 0, 'top_k': 25}
Best lags: {'admissions': [1, 2, 3, 4, 5, 6, 7, 14], 'calls': [1, 2, 3, 4, 5, 6, 7, 14]}


In [10]:
# forecast with the best parameters and best lags
mv_model = ml_mv_forecaster(model=LGBMRegressor(**best_params, verbose=-1),
              target_cols=['admissions', "calls"], lags = best_lags,
                cat_variables=cat_variables, categorical_encoder=ohe,
                difference={"admissions": 1, "calls": 1})
mv_model.fit(train)
mv_forecasts = mv_model.forecast(H=30, exog=test[cat_variables])
mv_forecasts

{'admissions': array([7927.77623633, 7979.85705744, 8047.84011864, 8075.97588743,
        8051.9219922 , 8058.83755567, 7991.42795185, 7950.6251427 ,
        7984.00010659, 8042.22632985, 8068.45140319, 8048.18706721,
        8034.65730985, 7976.58006664, 7951.55689959, 7977.90256632,
        8014.89117582, 8031.62212142, 8007.88397501, 7965.92344402,
        7899.82598425, 7878.39124364, 7922.71575572, 7958.31402347,
        7973.71685794, 7947.35494002, 7903.86148728, 7843.82945727,
        7822.3703652 , 7854.31693526]),
 'calls': array([1241.52487435, 1241.63676174, 1231.47489278, 1221.57119287,
        1222.58304612, 1231.62162295, 1238.13826939, 1236.11067467,
        1239.42920967, 1251.95697966, 1234.90405136, 1219.97517758,
        1223.12722736, 1232.73885405, 1230.03811079, 1243.99572587,
        1257.79014964, 1245.34852843, 1231.00112905, 1231.37679792,
        1247.2214911 , 1243.51602778, 1255.58619443, 1266.78873012,
        1254.08690619, 1239.01068276, 1239.76686463, 

### optuna_tune example for multivariate forecasting

In [11]:
from peshbeen.model_selection import mv_optuna_tune
lgb_param_space = {
    "learning_rate":     lambda t: t.suggest_float("learning_rate", 0.001, 0.6),
    "num_leaves":        lambda t: t.suggest_int("num_leaves", 10, 200),
    "max_depth":         lambda t: t.suggest_int("max_depth", 2, 18),
    "bagging_fraction":  lambda t: t.suggest_float("bagging_fraction", 0.5, 1.0),
    "feature_fraction":  lambda t: t.suggest_float("feature_fraction", 0.5, 1.0),
    "min_data_in_leaf":  lambda t: t.suggest_int("min_data_in_leaf", 5, 100),
    "lambda_l2":         lambda t: t.suggest_float("lambda_l2", 0.0, 10.0),
    "lambda_l1":         lambda t: t.suggest_float("lambda_l1", 0.0, 10.0),
    "other_rate":        lambda t: t.suggest_float("other_rate", 0.05, 0.3),
    "num_iterations":    lambda t: t.suggest_int("num_iterations", 30, 700),
    "top_k":             lambda t: t.suggest_int("top_k", 8, 30),
    "seed":              lambda t: 0,   # fixed, not sampled
    'lags': lambda t: t.suggest_categorical(
                             "lags", [1,2,3,4,5,
                                [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14]
                             ]),    
}

mv_model = ml_mv_forecaster(model=LGBMRegressor(verbose=-1),
              target_cols=['admissions', "calls"], lags = {"admissions": 7, "calls": 7},
                cat_variables=cat_variables, categorical_encoder=ohe,
                difference={"admissions": 1, "calls": 1})

best_params, best_lags = mv_optuna_tune(model=mv_model, df=train, target_col= "admissions", cv_split=5, step_size=10,
                                        test_size=30, eval_metric=RMSE, eval_num=4,
                                        param_space=lgb_param_space)

In [12]:
# forecast with the best parameters and best lags from optuna
mv_model = ml_mv_forecaster(model=LGBMRegressor(**best_params, verbose=-1),
              target_cols=['admissions', "calls"], lags = best_lags,
                cat_variables=cat_variables, categorical_encoder=ohe,
                difference={"admissions": 1, "calls": 1})
mv_model.fit(train)
mv_forecasts = mv_model.forecast(H=30, exog=test[cat_variables])
mv_forecasts

{'admissions': array([7935.14315257, 7970.90675656, 8002.92016615, 8048.90823678,
        8033.64840845, 8042.25186495, 7928.8171134 , 7866.72709409,
        7916.78333824, 7974.97838866, 8032.52400256, 8048.69191654,
        8028.22030219, 7922.8224261 , 7877.80159889, 7891.08607566,
        7902.60135165, 7930.38667538, 7922.40420477, 7875.10556851,
        7756.79506162, 7678.48076523, 7707.27255567, 7793.18368588,
        7839.84025797, 7863.59328481, 7833.28365494, 7793.27549119,
        7658.86778351, 7718.78182836]),
 'calls': array([1246.96794932, 1267.9680812 , 1247.47884294, 1259.16197046,
        1253.63753895, 1217.37889852, 1261.44166068, 1258.20260967,
        1278.37265773, 1311.06918846, 1281.89314461, 1234.14526272,
        1267.03836656, 1249.22227422, 1263.39943326, 1295.82796224,
        1290.89659145, 1282.04761665, 1272.93213389, 1272.51539136,
        1296.68488101, 1323.31202637, 1328.96935473, 1344.03842288,
        1301.3937191 , 1295.43326274, 1290.78330851, 